In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

sys.path.append(os.getcwd())

try:
    from retinal_fundus_disease_classifier import RetinalFundusDiseaseClassifier
except ImportError:
    pass

In [ ]:
# Configuration
BATCH_SIZE = 32
EPOCHS = 20
IMG_HEIGHT = 224
IMG_WIDTH = 224
INPUT_SHAPE = (IMG_HEIGHT, IMG_WIDTH, 3)

# Paths
ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), '../../..'))
DATASET_DIR = os.path.join(ROOT_DIR, 'ai/input/datasets/kaggle/rohitrawat25/combined-fundus-images')
CSV_PATH = os.path.join(DATASET_DIR, 'label_images.csv')
IMAGES_DIR = os.path.join(DATASET_DIR, 'images')
CHECKPOINT_DIR = os.path.join(ROOT_DIR, 'ai/models/checkpoints')

if not os.path.exists(CHECKPOINT_DIR):
    os.makedirs(CHECKPOINT_DIR)

print(f"Dataset CSV: {os.path.relpath(CSV_PATH, ROOT_DIR)}")
print(f"Images Dir: {os.path.relpath(IMAGES_DIR, ROOT_DIR)}")
print(f"Checkpoints Dir: {os.path.relpath(CHECKPOINT_DIR, ROOT_DIR)}")


In [ ]:
# Load Data
if os.path.exists(CSV_PATH):
    df = pd.read_csv(CSV_PATH)
    print(f"Loaded dataframe with {len(df)} samples.")
else:
    print(f"Error: CSV file not found at {CSV_PATH}")
    df = pd.DataFrame(columns=['images', 'label'])

print("Columns:", df.columns)
if 'label' in df.columns:
    labels = sorted(df['label'].unique())
    print(f"Unique labels ({len(labels)}): {labels}")
    NUM_CLASSES = len(labels)
else:
    print("Error: 'label' column not found.")
    NUM_CLASSES = 4

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

x_col = 'images'
if x_col not in df.columns and 'filepath' in df.columns:
    x_col = 'filepath'

print(f"Using x_col='{x_col}'")

if len(df) > 0:
    train_generator = train_datagen.flow_from_dataframe(
        dataframe=df,
        directory=IMAGES_DIR,
        x_col=x_col,
        y_col='label',
        target_size=(IMG_HEIGHT, IMG_WIDTH),
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        subset='training'
    )

    validation_generator = train_datagen.flow_from_dataframe(
        dataframe=df,
        directory=IMAGES_DIR,
        x_col=x_col,
        y_col='label',
        target_size=(IMG_HEIGHT, IMG_WIDTH),
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        subset='validation'
    )
else:
    train_generator = None
    validation_generator = None
    print("Skipping generator creation: No data.")

In [ ]:
model = RetinalFundusDiseaseClassifier._disease_classify_network(
    input_shape=INPUT_SHAPE,
    num_classes=NUM_CLASSES
)

model.summary()

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Recall(name='recall'), tf.keras.metrics.Precision(name='precision')]
)

checkpoint_path = os.path.join(CHECKPOINT_DIR, 'disease_classify_model_nb_1.h5')
print(f"Model will be saved to: {os.path.relpath(checkpoint_path, ROOT_DIR)}")

checkpoint = ModelCheckpoint(
    checkpoint_path,
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

if train_generator and validation_generator:
    history = model.fit(
        train_generator,
        epochs=EPOCHS,
        validation_data=validation_generator,
        callbacks=[checkpoint, early_stopping, reduce_lr]
    )

    final_model_path = os.path.join(CHECKPOINT_DIR, 'disease_classify_model_nb_1.h5')
    model.save(final_model_path)
    print("Training pipeline finished.")
else:
    print("Training skipped due to missing data generators.")
